# Step 1: Get Features From Multiple Datasets
- Using [pybiber](https://pypi.org/project/pybiber/)

In [1]:
# HuggingFace Login
import os
from dotenv import load_dotenv
from huggingface_hub import login
load_dotenv()
hf_token = os.getenv("HUGGING_FACE_TOKEN")
login(hf_token)

from transformers import logging
logging.set_verbosity_error()

In [2]:
import polars as pl
import os
import numpy as np
import random
import torch
import pandas as pd
import gc
import csv
from transformers import pipeline

In [ ]:
DEVICE = 0 if torch.cuda.is_available() else -1
FILE_PATH = 'getText/datasetsPrep'
OUTPUT_DIR = 'zeroShotOutputs'
BATCH_SIZE = 4

In [4]:
ZERO_SHOT_MODELS = [
    "cross-encoder/nli-deberta-v3-small", # low capacity
    # "cross-encoder/nli-MiniLM2-L6-H768",
    # "MoritzLaurer/DeBERTa-v3-xsmall-mnli-fever-anli-ling-binary",
    # "MoritzLaurer/roberta-base-zeroshot-v2.0-c", # not a large amount of improvement after adding these models so excluding them
    "typeform/distilbert-base-uncased-mnli", # medium capacity
    "valhalla/distilbart-mnli-12-3", # higher capacity
]

# Exact mapping taken from https://www.uni-bamberg.de/fileadmin/eng-ling/fs/Chapter_21/23DimensionsofEnglish.html which has extracted the same from Biber and Conrad's Variation in English (https://doi.org/10.4324/9781315840888)
BIBER_LABEL_MAP = {
    "factor_1": {
        "informational, dense, precise": -1,
        "involved, interactive, affective": 1
    },
    "factor_2": {
        "non-narrative, expository, informational": -1,
        "narrative, event-focused, storytelling": 1
    },
    "factor_3": {
        "situation-dependent, context-bound, implicit": -1,
        "explicit, context-independent, elaborated": 1
    },
    "factor_4": {
        "non-persuasive, non-argumentative, neutral": -1,
        "persuasive, argumentative, modalized": 1
    },
    "factor_5": {
        "non-abstract, concrete, human-centered": -1,
        "abstract, impersonal, technical": 1
    },
    "factor_6": {
        "compressed, dense, clause-poor": -1,
        "elaborated, expanded, clause-rich": 1
    }
}

# Using different prompt templates increases robustness. Ability to modify for different factors. 
TEMPLATES = { # [default, default, more specialized, even more specialized, most specialised], moving from neutral to theory-aware phrasing
    "factor_1": ["This example is {}.", "This text is {}.", "This text is written in a {} style.", "The writing style of this text is {}.", "This text shows {} characteristics."],
    "factor_2": ["This example is {}.", "This text is {}.", "This text is written in a {} style.", "The writing style of this text is {}.", "This text shows {} characteristics."],
    "factor_3": ["This example is {}.", "This text is {}.", "This text is written in a {} style.", "The writing style of this text is {}.", "This text shows {} characteristics."],
    "factor_4": ["This example is {}.", "This text is {}.", "This text is written in a {} style.", "The writing style of this text is {}.", "This text shows {} characteristics."],
    "factor_5": ["This example is {}.", "This text is {}.", "This text is written in a {} style.", "The writing style of this text is {}.", "This text shows {} characteristics."],
    "factor_6": ["This example is {}.", "This text is {}.", "This text is written in a {} style.", "The writing style of this text is {}.", "This text shows {} characteristics."]
}


In [5]:
# Set random states.
def set_random_states(random_state):
    # Set various random seeds.
    np.random.seed(random_state)
    pl.set_random_seed(random_state)
    pd.core.common.random_state(random_state)
    random.seed(random_state)
    torch.manual_seed(random_state)
    torch.cuda.manual_seed_all(random_state)
    os.environ["PYTHONHASHSEED"] = str(random_state)
    os.environ["TOKENIZERS_PARALLELISM"] = "false"
    try:
        torch.use_deterministic_algorithms(True)
    except Exception:
        pass
    return random_state

RANDOM_STATE = set_random_states(1618)

In [6]:
def make_dfs(df, output_dir, output_file_name):
    os.makedirs(f"./{output_dir}/", exist_ok=True)
    # Save CSV.
    df.to_csv(f"./{output_dir}/{output_file_name}.csv", index=False, lineterminator="\n", quoting=csv.QUOTE_ALL)

    # Save JSON
    df.to_json(f"./{output_dir}/{output_file_name}.json", orient="records", indent=2)

In [ ]:
# Load data.
list_of_dfs = []
for folder in os.listdir(f'./{FILE_PATH}'):
    if os.path.isdir(f'./{FILE_PATH}/{folder}'):
        for file in os.listdir(f'./{FILE_PATH}/{folder}'):
            if file.endswith(".csv"):
                temp_file_path = f'./{FILE_PATH}/{folder}/{file}'
                temp_tag = file.replace('_train.csv', '')
                temp_df = pl.read_csv(temp_file_path)
                temp_df = (
                    temp_df
                    .with_row_index("index_num") # , offset=1) if you want to start index from 1
                    .with_columns(
                        (pl.lit(temp_tag) + "_" + pl.col("index_num").cast(pl.Utf8)).alias("doc_id")
                    )).select(['text', 'doc_id'])
                
                # Remove invalid data.
                temp_df = temp_df.with_columns(
                    pl.col("doc_id").str.split("_").list.get(0).alias("tag")
                )
                print(temp_df.group_by("tag").len())
                print(f"Total Texts Before Empty String Removal: {len(temp_df)}")

                temp_df = temp_df.with_columns(pl.col("text").str.strip_chars().alias("text")).filter(pl.col("text").is_not_null() & (pl.col("text") != ""))

                temp_df = temp_df.with_columns(
                    pl.col("doc_id").str.split("_").list.get(0).alias("tag")
                )
                print(temp_df.group_by("tag").len())
                print(f"Total Texts After Empty String Removal: {len(temp_df)}")

                # Set up df for use.
                df = pl.DataFrame({
                    "doc_id": temp_df['doc_id'].to_list(),
                    "text": temp_df['text'].to_list()
                })

                # Light preprocessing to strip extra whitespace.
                df = df.with_columns(
                    pl.col("text")
                    .str.strip_chars()
                    .str.replace_all(r"\s+", " ")
                    .str.replace_all(r"^\s*-\s*", "") # Remove dashes at the beginning of texts.
                    .str.replace_all(r"^\s*\d+\.\s*", "") # Remove numbers in 1., 2., 3. format at the beginning of the text. 
                )
                temp_df = temp_df.to_pandas()
                texts = temp_df['text'].values.tolist()
                doc_ids = temp_df['doc_id'].values.tolist()

                assert len(texts) == len(doc_ids), "texts and doc_ids are not of the same length."
                print(f"Using {len(texts)} train datapoints...")
                rows = []
                for model_name in ZERO_SHOT_MODELS:
                    classifier = pipeline(
                        "zero-shot-classification",
                        model=model_name,
                        device=DEVICE
                    )
                    temp_factors_list = [{'model_name': model_name, 'doc_id': doc_id} for doc_id in doc_ids]
                    for factor, description in BIBER_LABEL_MAP.items():
                        factor_scores = [0.0 for _ in texts]
                        for template in TEMPLATES[factor]:
                            outputs = classifier(
                                texts,
                                candidate_labels=list(description.keys()),
                                hypothesis_template=template,
                                multi_label = True,
                                batch_size = BATCH_SIZE
                            )
                            # Ensure outputs is a list.
                            if isinstance(outputs, dict):
                                outputs = [outputs]
                            # Accumulate weighted scores per text.
                            for i, output in enumerate(outputs):
                                score = sum(description[label] * s for label, s in zip(output['labels'], output['scores']))
                                factor_scores[i] += score
                        # Average over templates
                        factor_scores = [s / len(TEMPLATES[factor]) for s in factor_scores]

                        # Assign to temp_factors_list
                        for i, score in enumerate(factor_scores):
                            temp_factors_list[i][factor] = score

                    rows.extend(temp_factors_list)

                    # Free memory.
                    del classifier
                    torch.cuda.empty_cache()
                    gc.collect()

                df = pd.DataFrame(rows)
                # Simple averaging will prevent over-confidence. 
                mean_scores = df.drop(columns='model_name').groupby("doc_id").mean().reset_index()

                make_dfs(df, f"{OUTPUT_DIR}/{folder}", f"{file.replace('.csv', '')}AllModels")
                make_dfs(mean_scores, f"{OUTPUT_DIR}/{folder}", f"{file.replace('.csv', '')}MeanScores")

                print(pl.from_pandas(df))
                print(pl.from_pandas(mean_scores))



shape: (1, 2)
┌───────────────┬─────┐
│ tag           ┆ len │
│ ---           ┆ --- │
│ str           ┆ u32 │
╞═══════════════╪═════╡
│ dementiaAudio ┆ 383 │
└───────────────┴─────┘
Total Texts Before Empty String Removal: 383
shape: (1, 2)
┌───────────────┬─────┐
│ tag           ┆ len │
│ ---           ┆ --- │
│ str           ┆ u32 │
╞═══════════════╪═════╡
│ dementiaAudio ┆ 383 │
└───────────────┴─────┘
Total Texts After Empty String Removal: 383
Using 16 train datapoints...


Loading weights:   0%|          | 0/106 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/283 [00:00<?, ?it/s]

shape: (48, 8)
┌─────────────┬─────────────┬──────────┬───────────┬───────────┬───────────┬───────────┬───────────┐
│ model_name  ┆ doc_id      ┆ factor_1 ┆ factor_2  ┆ factor_3  ┆ factor_4  ┆ factor_5  ┆ factor_6  │
│ ---         ┆ ---         ┆ ---      ┆ ---       ┆ ---       ┆ ---       ┆ ---       ┆ ---       │
│ str         ┆ str         ┆ f64      ┆ f64       ┆ f64       ┆ f64       ┆ f64       ┆ f64       │
╞═════════════╪═════════════╪══════════╪═══════════╪═══════════╪═══════════╪═══════════╪═══════════╡
│ cross-encod ┆ dementiaAud ┆ 0.818035 ┆ 0.454327  ┆ -0.755611 ┆ 0.321343  ┆ -0.796742 ┆ 0.867058  │
│ er/nli-debe ┆ io_0        ┆          ┆           ┆           ┆           ┆           ┆           │
│ rta-v3-s…   ┆             ┆          ┆           ┆           ┆           ┆           ┆           │
│ cross-encod ┆ dementiaAud ┆ 0.711176 ┆ -0.231157 ┆ -0.971935 ┆ 0.293562  ┆ -0.596696 ┆ 0.755122  │
│ er/nli-debe ┆ io_1        ┆          ┆           ┆           ┆           ┆

...

shape: (48, 8)
┌──────────────┬──────────────┬──────────┬──────────┬───────────┬───────────┬───────────┬──────────┐
│ model_name   ┆ doc_id       ┆ factor_1 ┆ factor_2 ┆ factor_3  ┆ factor_4  ┆ factor_5  ┆ factor_6 │
│ ---          ┆ ---          ┆ ---      ┆ ---      ┆ ---       ┆ ---       ┆ ---       ┆ ---      │
│ str          ┆ str          ┆ f64      ┆ f64      ┆ f64       ┆ f64       ┆ f64       ┆ f64      │
╞══════════════╪══════════════╪══════════╪══════════╪═══════════╪═══════════╪═══════════╪══════════╡
│ cross-encode ┆ dementiaAudi ┆ 0.99074  ┆ 0.265091 ┆ -0.932553 ┆ -0.016553 ┆ -0.663021 ┆ 0.242292 │
│ r/nli-debert ┆ o_val.csv_0  ┆          ┆          ┆           ┆           ┆           ┆          │
│ a-v3-s…      ┆              ┆          ┆          ┆           ┆           ┆           ┆          │
│ cross-encode ┆ dementiaAudi ┆ 0.910014 ┆ 0.217831 ┆ -0.83209  ┆ 0.193127  ┆ -0.766217 ┆ 0.844791 │
│ r/nli-debert ┆ o_val.csv_1  ┆          ┆          ┆           ┆           

Loading weights:   0%|          | 0/106 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/283 [00:00<?, ?it/s]

shape: (48, 8)
┌─────────────┬─────────────┬──────────┬───────────┬───────────┬───────────┬───────────┬───────────┐
│ model_name  ┆ doc_id      ┆ factor_1 ┆ factor_2  ┆ factor_3  ┆ factor_4  ┆ factor_5  ┆ factor_6  │
│ ---         ┆ ---         ┆ ---      ┆ ---       ┆ ---       ┆ ---       ┆ ---       ┆ ---       │
│ str         ┆ str         ┆ f64      ┆ f64       ┆ f64       ┆ f64       ┆ f64       ┆ f64       │
╞═════════════╪═════════════╪══════════╪═══════════╪═══════════╪═══════════╪═══════════╪═══════════╡
│ cross-encod ┆ dementiaAud ┆ 0.884931 ┆ 0.780872  ┆ -0.614039 ┆ 0.51456   ┆ -0.617878 ┆ 0.643679  │
│ er/nli-debe ┆ io_test.csv ┆          ┆           ┆           ┆           ┆           ┆           │
│ rta-v3-s…   ┆ _0          ┆          ┆           ┆           ┆           ┆           ┆           │
│ cross-encod ┆ dementiaAud ┆ 0.573807 ┆ -0.402552 ┆ -0.583798 ┆ 0.077131  ┆ -0.595831 ┆ 0.988435  │
│ er/nli-debe ┆ io_test.csv ┆          ┆           ┆           ┆           ┆